In [ ]:
import json
import torch

from dataset import MetaStyleDataset
from encoder import TransformerEncoder

from torch.utils.data import DataLoader
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def unpack_batch(batch):
    return (
        batch['support_pos'].to(device),
        batch['support_mask'].to(device),
        batch['support_labels'].to(device),
        batch['query_pos'].to(device),
        batch['query_mask'].to(device),
        batch['query_labels'].to(device)
    )

def plot_data(within_players, between_players, self_non, title, data_num):
    plt.figure(figsize=(6, 4))
    plt.hist(within_players.cpu(), bins=30, density=True, alpha=0.5, color='C0', label="within players")
    plt.hist(between_players.cpu(), bins=30, density=True, alpha=0.5, color='C1', label="between players")
    plt.hist(self_non.cpu(), bins=30, density=True, alpha=0.5, color='C2', label="between self and non-self")
    plt.xlabel("Cosine similarity")
    plt.ylabel("Density")
    plt.title(f"Similarity Distribution - {title} - {data_num}")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_embedding_pca(embeddings, labels, title, data_num):
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import numpy as np

    pca = PCA(n_components=2)
    reduced_embeddings = pca.fit_transform(embeddings.cpu().numpy())

    unique_labels = torch.unique(labels).cpu().numpy()
    num_labels = len(unique_labels)

    cmap = plt.get_cmap('tab20', num_labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, lab in enumerate(unique_labels):
        indices = (labels == lab).cpu().numpy()  # 得到当前 player 的索引布尔数组
        ax.scatter(
            reduced_embeddings[indices, 0],
            reduced_embeddings[indices, 1],
            color=cmap(i),
            label=f"Player {lab}",
            alpha=0.5
        )
    ax.set_title(f"PCA of Embeddings - {title} - {data_num}")
    ax.set_xlabel("PCA Component 1")
    ax.set_ylabel("PCA Component 2")
    ax.legend(title="Player ID")
    plt.tight_layout()
    plt.show()

def load_dataset_file(path):
    with open(f"chess_data_parse/{path}.json", "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
max_len = 100

encoder = TransformerEncoder(cnn_in_channels=224, state_embed_dim=256, transformer_d_model=256,
                                        num_heads=8, num_layers=3, dropout=0.1, max_seq_len=max_len).to(device)

d = torch.load('./models/trained_model/player_encoder_60.pt')
encoder.load_state_dict(d["model_state_dict"])

test_dataset = MetaStyleDataset(load_dataset_file("test_players"), 150, max_len=max_len)

test_loader = DataLoader(test_dataset,
                        batch_size=4,
                        shuffle=True,
                        pin_memory=False,
                        num_workers=4,
                        persistent_workers=True)

train_dataset = MetaStyleDataset(load_dataset_file("train_players"), 150, K=10, Q=10, max_len=max_len)

train_loader = DataLoader(train_dataset,
                        batch_size=4,
                        shuffle=True,
                        pin_memory=False,
                        num_workers=4,
                        persistent_workers=True)

In [ ]:
prototypes = []
prototype_ids = []  # 每个 prototype 对应的 player_id
prototype_labels = []  # 每个 prototype 对应的 player_id

query_sims = []             # self 相似度
retrieval_labels = []      # GT label
retrieval_preds = []       # 预测的 top-1 prototype 的 player_id
non_self_sims = []         # query 和其他玩家 prototype 的平均相似度
all_query_embeddings = []
all_query_embeddings_labels = []

print(f"Test dataset size: {len(test_loader.dataset)}")
print(f"Train dataset size: {len(train_loader.dataset)}")

batch_idx = 0
for batch in test_loader:
    support_pos, support_mask, support_labels, query_pos, query_mask, query_labels = unpack_batch(batch)
    B = support_pos.shape[0]
    batch_idx += 1

    with torch.no_grad():
        for i in range(B):
            # --- Support set ---
            task_support_pos = support_pos[i]      # [25, ...]
            task_support_mask = support_mask[i]
            task_support_labels = support_labels[i].to(device)

            all_support_embeddings = []
            for j in range(task_support_pos.shape[0]):
                pos = task_support_pos[j].unsqueeze(0).to(device)
                mask = task_support_mask[j].unsqueeze(0).to(device)

                with torch.autocast(device_type="cuda"):
                    _, emb = encoder(pos, mask)
                    all_support_embeddings.append(emb.squeeze(0))  # [D]
            all_support_embeddings = torch.stack(all_support_embeddings, dim=0)  # [25, D]

            # --- Query set ---
            task_query_pos = query_pos[i]
            task_query_mask = query_mask[i]
            task_query_labels = query_labels[i].to(device)

            query_embeddings = []
            for j in range(task_query_pos.shape[0]):
                pos = task_query_pos[j].unsqueeze(0).to(device)
                mask = task_query_mask[j].unsqueeze(0).to(device)

                with torch.autocast(device_type="cuda"):
                    _, emb = encoder(pos, mask)
                    query_embeddings.append(emb.squeeze(0))
            query_embeddings = torch.stack(query_embeddings, dim=0)  # [Q, D]
            all_query_embeddings.append(query_embeddings)
            all_query_embeddings_labels.append(task_query_labels + 1000 * (batch_idx * B + i))
            print(f"task_query_labels + 5 * (batch_idx * B + i)", task_query_labels + 5 * (batch_idx * B + i))

            # --- Per player ---
            unique_players = task_support_labels.unique()

            for player_id in unique_players:
                pid = player_id.item()

                # Build prototype
                mask_support = task_support_labels == player_id
                proto = all_support_embeddings[mask_support].mean(dim=0)  # [D]
                prototypes.append(proto)
                prototype_ids.append(pid)
                prototype_labels.append(pid + 1000 * (batch_idx * B + i))

                # Find query embeddings for this player
                mask_query = task_query_labels == player_id
                if mask_query.sum() == 0:
                    continue
                query_emb = query_embeddings[mask_query]  # [Q, D]

                # 1. Self similarity
                sim = F.cosine_similarity(query_emb, proto.unsqueeze(0), dim=1)  # [Q]
                avg_sim = sim.mean().item()
                query_sims.append(avg_sim)

                # 2. Retrieval & Soft Matching
                for emb in query_emb:
                    retrieval_labels.append(pid)

                    all_sims = F.cosine_similarity(
                        emb.unsqueeze(0),                      # [1, D]
                        torch.stack(prototypes).to(device),   # [N, D]
                        dim=1
                    )  # [N]

                    all_ids_tensor = torch.tensor(prototype_ids, device=all_sims.device)
                    top1_idx = torch.argmax(all_sims).item()
                    pred_id = prototype_ids[top1_idx]
                    retrieval_preds.append(pred_id)

                    # 3. Non-self prototype similarity
                    mask_not_self = all_ids_tensor != pid
                    if mask_not_self.sum() > 0:
                        non_self_avg = all_sims[mask_not_self].mean().item()
                        non_self_sims.append(non_self_avg)

# --- Evaluation Metrics ---

# (1) prototype-prototype similarity
prototypes_tensor = torch.stack(prototypes, dim=0)
N = prototypes_tensor.shape[0]
sim_matrix = F.cosine_similarity(
    prototypes_tensor.unsqueeze(1), prototypes_tensor.unsqueeze(0), dim=2
)
mask = ~torch.eye(N, dtype=torch.bool, device=prototypes_tensor.device)
pairwise_sim = sim_matrix[mask]
avg_similarity = pairwise_sim.mean().item()
print(f"Average similarity between prototypes: {avg_similarity:.4f}")

# (2) query-prototype (self) similarity
avg_query_sim = sum(query_sims) / len(query_sims)
print(f"Average similarity between query and prototype: {avg_query_sim:.4f}")

# (3) retrieval accuracy
correct = sum([pred == gt for pred, gt in zip(retrieval_preds, retrieval_labels)])
retrieval_acc = correct / len(retrieval_labels)
print(f"Top-1 retrieval accuracy: {retrieval_acc:.4f}")

# (4) query and non-self prototype similarity
avg_non_self_sim = sum(non_self_sims) / len(non_self_sims)
print(f"Average similarity between query and non-self prototypes: {avg_non_self_sim:.4f}")


all_query_embeddings = torch.cat([emb for emb in all_query_embeddings], dim=0)  # Ensure it's a list of tensors
all_query_embeddings_labels = torch.cat([lab for lab in all_query_embeddings_labels], dim=0)
all_prototypes_labels = torch.tensor(prototype_labels, device=all_query_embeddings.device)

Test dataset size: 150
Train dataset size: 150
task_query_labels + 5 * (batch_idx * B + i) tensor([20, 20, 20, 20, 20, 21, 21, 21, 21, 21, 22, 22, 22, 22, 22, 23, 23, 23,
        23, 23, 24, 24, 24, 24, 24], device='cuda:0')
task_query_labels + 5 * (batch_idx * B + i) tensor([25, 25, 25, 25, 25, 26, 26, 26, 26, 26, 27, 27, 27, 27, 27, 28, 28, 28,
        28, 28, 29, 29, 29, 29, 29], device='cuda:0')
task_query_labels + 5 * (batch_idx * B + i) tensor([30, 30, 30, 30, 30, 31, 31, 31, 31, 31, 32, 32, 32, 32, 32, 33, 33, 33,
        33, 33, 34, 34, 34, 34, 34], device='cuda:0')
task_query_labels + 5 * (batch_idx * B + i) tensor([35, 35, 35, 35, 35, 36, 36, 36, 36, 36, 37, 37, 37, 37, 37, 38, 38, 38,
        38, 38, 39, 39, 39, 39, 39], device='cuda:0')
task_query_labels + 5 * (batch_idx * B + i) tensor([40, 40, 40, 40, 40, 41, 41, 41, 41, 41, 42, 42, 42, 42, 42, 43, 43, 43,
        43, 43, 44, 44, 44, 44, 44], device='cuda:0')
task_query_labels + 5 * (batch_idx * B + i) tensor([45, 45, 45,

In [ ]:
plot_data(torch.tensor(query_sims), pairwise_sim, torch.tensor(non_self_sims), "Test set", "K=10, Q=10")

In [ ]:
print(f"Number of prototypes: {len(prototypes)}")
print(f"Number of query embeddings: {all_query_embeddings.shape[0]}, {all_query_embeddings_labels.shape[0]}, {all_prototypes_labels.shape[0]}")

all_embeddings = torch.cat([prototypes_tensor.cpu(), all_query_embeddings.cpu()], dim=0)
all_labels = torch.cat([all_query_embeddings_labels.cpu(), all_prototypes_labels.cpu()], dim=0)

unique_players = torch.unique(all_labels)
print("Total unique players:", len(unique_players))  # 例如 150

selected_indices = torch.randperm(len(unique_players))[:5]
selected_players = unique_players[selected_indices]
print("Selected players:", selected_players)

mask = torch.zeros_like(all_labels, dtype=torch.bool)
for pid in selected_players:
    mask |= (all_labels == pid)

filtered_embeddings = all_embeddings[mask]
filtered_labels = all_labels[mask]

print("Filtered embeddings shape:", filtered_embeddings.shape)
print("Filtered labels shape:", filtered_labels.shape)

plot_embedding_pca(filtered_embeddings, filtered_labels, "Test", "5 players, K=10, Q=10")

Number of prototypes: 750
Number of query embeddings: 3750, 3750, 750
Total unique players: 740
Selected players: tensor([147002,  92000,   4000,  87003,  67004])
Filtered embeddings shape: torch.Size([30, 256])
Filtered labels shape: torch.Size([30])
